# LIME #

## Data Pipeline ##



In [1]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if device.type == 'cuda':
  print(f'Running on GPU: {torch.cuda.get_device_name(0)}')
else:
  print('Running on CPU')

Running on GPU: NVIDIA GeForce RTX 2080 Super with Max-Q Design


### Import Dataset ###

In [2]:
import pandas as pd

# path of each of the three csv files of the GoEmotions dataset
# Assuming the files are directly in MyDrive based on the file list
csv_path1 = 'C:/Users/maewa/Coding/IntProjectII/IMCS3020U_Project_TBA/data/goemotions_1.csv'
csv_path2 = 'C:/Users/maewa/Coding/IntProjectII/IMCS3020U_Project_TBA/data/goemotions_2.csv'
csv_path3 = 'C:/Users/maewa/Coding/IntProjectII/IMCS3020U_Project_TBA/data/goemotions_3.csv'

# List of the three files
csv_files = [csv_path1, csv_path2, csv_path3]

# generate DataFrame from each file then concatenate all three DataFrames into one
df = pd.concat((pd.read_csv(filename) for filename in csv_files), ignore_index=True)

#isolate the columns for emotion labels
label_cols = df.columns[9:].tolist()
#print(df.columns)
#print(label_cols)

# turn text column into lists
texts = df['text'].tolist()
# results in a list of lists representing the labels
labels = df[label_cols].values.tolist()

### Split Data into Train and Test ###

In [3]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.1, random_state=42
)

## Tokenization ##

### Load BERT Tokenizer ###

In [4]:
from transformers import AutoTokenizer

MODEL_NAME = 'bert-base-uncased'

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f'Tokenizer type loaded: {type(tokenizer).__name__}')

c:\Users\maewa\anaconda3\envs\bert_xai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokenizer type loaded: BertTokenizer


### Tokenize Train and Test Sets ###

In [5]:
# Tokenize train text
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding='max_length',
    max_length=128,
)

#Tokenize validation text
val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding='max_length',
    max_length=128,
)

print("Tokenization Example (First Text)")
print("Input text:", train_texts[0])
print("Input IDs (Partial):", train_encodings['input_ids'][0][:10], "...")
print("Attention Mask (partial):", train_encodings['attention_mask'][0][:10], '...')

Tokenization Example (First Text)
Input text: the longest week (2014) I don’t know if it’s still available but it was pretty good romantic comedy
Input IDs (Partial): [101, 1996, 6493, 2733, 1006, 2297, 1007, 1045, 2123, 1521] ...
Attention Mask (partial): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1] ...


## Create Pytorch Dataset ##

In [6]:
class GoEmotionsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __getitem__(self,idx):
        # converts the encodings lists to Pytorch tensors
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}

        # Add multi-hot encoded labels
        item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

### Instantiate Dataset ###

In [7]:
train_dataset = GoEmotionsDataset(train_encodings, train_labels)
val_dataset = GoEmotionsDataset(val_encodings, val_labels)

### Set up PyTorch DataLoader ###

In [8]:
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler

# Set batch size here
BATCH_SIZE = 16

# Training DataLoader
train_dataloader = DataLoader(
    train_dataset,
    # RandomSampler is used to shuffle data during each epoch
    sampler=RandomSampler(train_dataset),
    batch_size=BATCH_SIZE
)

# Validation DataLoader
val_dataloader = DataLoader(
    val_dataset,
    #SequentialSampler is used to ensure the data order is predictable
    sampler=SequentialSampler(val_dataset),
    batch_size=BATCH_SIZE
)

print(f'Train Dataloader created, batch size is {BATCH_SIZE}')
print(f'Validation Dataloader created with a batch size of {BATCH_SIZE}')

Train Dataloader created, batch size is 16
Validation Dataloader created with a batch size of 16


## Define Model ##

In [9]:
from transformers import AutoModelForSequenceClassification, AutoConfig

NUM_LABELS = 28

# configuration for multilabel classification
config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    config=config
)

# set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if device.type == 'cuda':
  print(f"Running on GPU: {torch.cuda.get_device_name(0)}")
else:
  print("Running on CPU (GPU not available or selected)")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 480.69it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those pa

Running on GPU: NVIDIA GeForce RTX 2080 Super with Max-Q Design


### Import Pre-Trained Model Weights ###

In [11]:
weights_path = "../weights/model_weights.pth"
state_dict = torch.load(weights_path, map_location=device)

model.load_state_dict(state_dict)

model.to(device)

model.eval()

C:\Users\maewa\AppData\Local\Temp\ipykernel_35556\689295023.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weights_path, map_location=device)


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,